# 01 — reproduce the selected exact-silver v1 training run

This is the canonical grader notebook. It contains the frozen
data payload, validates all release hashes, and trains only the
selected Complete-600 adapter. No input file or Google Drive is
required. Use an L4 or A100 runtime.

Expected core training time is roughly 6–12 minutes after
installation and model download. The original run used 75
optimizer steps and 600 deterministic row exposures.

## 1. Install the pinned environment

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q --upgrade --no-cache-dir "unsloth==2026.7.2" unsloth_zoo "datasets==4.3.0" "transformers==4.56.2" "trl==0.22.2" peft accelerate bitsandbytes

Restart the Colab runtime after installation, then run every remaining cell in order.

In [ ]:
from pathlib import Path
from collections import Counter
import base64
import gc
import hashlib
import io
import json
import math
import re
import shutil
import time
import zipfile

import torch
import unsloth
from datasets import Dataset
from google.colab import drive, files
from unsloth import FastLanguageModel

assert torch.cuda.is_available(), "Enable a GPU runtime first."
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
assert GPU_MEMORY_GB >= 20, (
    f"Use an L4/A100-class GPU; got {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)."
)

DATA_PAYLOAD_B64 = """..."""
EXPECTED_DATA_PAYLOAD_SHA256 = "2ad66a817804220bc23bf182d08aef2b95b711e90d63214d79f388c87ef47e94"
payload_bytes = base64.b64decode("".join(DATA_PAYLOAD_B64.split()))
assert hashlib.sha256(payload_bytes).hexdigest() == EXPECTED_DATA_PAYLOAD_SHA256
DATA_ROOT = Path("/content/exact_silver_v1_data")
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(payload_bytes)) as archive:
    archive.extractall(DATA_ROOT)

# Graders need no Drive input. Enable this only to carry the trained
# adapter automatically into notebook 02 or notebook 03.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/uae_adab_slm")
else:
    OUTPUT_ROOT = Path("/content/uae_adab_submission_outputs")
RUNS = OUTPUT_ROOT / "runs_v2_exact"
METRICS = OUTPUT_ROOT / "metrics_v2_exact"
for path in (RUNS, METRICS):
    path.mkdir(parents=True, exist_ok=True)

if "_EXACT_SESSION_NONCE_SHA256" not in globals():
    _EXACT_SESSION_NONCE_SHA256 = hashlib.sha256(
        f"{time.time_ns()}|{GPU_NAME}".encode("utf-8")
    ).hexdigest()
if "_EXACT_TRAINING_SESSION_RUN" not in globals():
    _EXACT_TRAINING_SESSION_RUN = None
print("GPU:", GPU_NAME, f"({GPU_MEMORY_GB:.1f} GB)")
print("Embedded data SHA-256:", EXPECTED_DATA_PAYLOAD_SHA256)
print("Output root:", OUTPUT_ROOT)

## 2. Validate the embedded exact-silver release

In [ ]:
DATA_PATHS = {
    "grounded120_train": DATA_ROOT / "grounded_120_train.jsonl",
    "grounded120_validation": DATA_ROOT / "grounded_120_validation.jsonl",
    "complete600_train": DATA_ROOT / "complete_600_train.jsonl",
    "complete600_validation": DATA_ROOT / "complete_600_validation.jsonl",
}
EXPECTED_ROWS = {
    "grounded120_train": 108,
    "grounded120_validation": 12,
    "complete600_train": 540,
    "complete600_validation": 60,
}
MANIFEST_PATH = DATA_ROOT / "exact_silver_release_manifest.json"
FILE_PATHS = {"exact_silver_release_manifest": MANIFEST_PATH, **DATA_PATHS}

# Prefilled from the frozen local release. A renamed, edited, or partial file
# fails before model loading.
EXPECTED_FILE_SHA256 = {
    "complete600_train": "990cdc7ca494a4e12efa1cc7a739030ef1412e1dd1a350f91b1d453e49a756a0",
    "complete600_validation": "cf10c2e316c13da2e2ebf1c1edee18ebfd3784085440a00b8a8c31e71fa0ec1c",
    "exact_silver_release_manifest": "ef8b907a497444b83410bd2fdcff37c4eb3922d90e485638303b8e5342ba662b",
    "grounded120_train": "591d6466dcda9b5aafdb1e19a51ba873c72a657e38de262e76f49d46bf37c19b",
    "grounded120_validation": "28470b409c5db194dc97a34b637a1a87275ed3b685869fc52dca3c0f7b6f56a5"
}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

assert set(EXPECTED_FILE_SHA256) == set(FILE_PATHS)
observed_file_sha256 = {}
for key, path in FILE_PATHS.items():
    assert path.exists(), f"Upload {path.name} to /content."
    expected = EXPECTED_FILE_SHA256[key].lower()
    assert re.fullmatch(r"[0-9a-f]{64}", expected)
    observed = sha256_file(path)
    assert observed == expected, (key, "external hash-lock mismatch")
    observed_file_sha256[key] = observed

release_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert release_manifest["schema_version"] == "uae_adab_exact_silver_600_release.v1"
assert release_manifest["release_authority"] is False
assert release_manifest["counts"] == {
    "grounded": {"all": 120, "train": 108, "validation": 12},
    "revised": {"all": 480, "train": 432, "validation": 48},
    "complete": {"all": 600, "train": 540, "validation": 60},
}
assert release_manifest["composition"]["source_grounded_fraction"] == 0.20
assert release_manifest["composition"]["train_source_grounded_fraction"] == 0.20
assert release_manifest["composition"]["validation_source_grounded_fraction"] == 0.20
release_manifest_sha256 = observed_file_sha256["exact_silver_release_manifest"]
for key, path in DATA_PATHS.items():
    artifact = release_manifest["artifacts"][key]
    assert artifact["file"] == path.name
    assert artifact["rows"] == EXPECTED_ROWS[key]
    assert observed_file_sha256[key] == artifact["sha256"], (key, "manifest hash mismatch")

assert release_manifest["subset_audit"]["same_ids_hashes_splits_and_groups"] is True
assert release_manifest["grouping"]["train_validation_group_overlap"] == 0
assert release_manifest["grouping"]["train_validation_source_family_overlap"] == 0
assert release_manifest["grouping"]["train_validation_id_overlap"] == 0
leakage = release_manifest["heldout"]["evaluation_leakage"]
assert leakage["exact_overlap_rows"] == []
assert leakage["twelve_gram_overlap_rows"] == []
print("FIVE-FILE SILVER HASH LOCK VALID", json.dumps(observed_file_sha256, indent=2))
print("IMPORTANT: this is the manifest-disclosed experimental silver release, not a gold corpus.")

In [ ]:
SYSTEM_MESSAGE = "<uae_adab_tutor>default</uae_adab_tutor>"

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line]

def row_id(row):
    value = row.get("compiler_assignment_id") or row.get("id")
    assert isinstance(value, str) and value
    return value

def canonical_hash(value):
    return hashlib.sha256(
        json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()

def canonical_tier(row):
    freeze = row.get("silver_freeze")
    if isinstance(freeze, dict) and isinstance(freeze.get("tier"), str):
        return freeze["tier"]
    quality = row.get("quality", {})
    if isinstance(quality, dict):
        tier = quality.get("silver_tier") or quality.get("review_tier")
        if isinstance(tier, str) and tier:
            return tier
    return {
        "accepted_expedited_silver": "reviewed_accepted_expedited",
        "accepted_unanimous_review": "reviewed_accepted_dual",
        "accepted_grounded_v3": "reviewed_accepted_grounded_v3",
        "candidate_unreviewed": "unreviewed_structural_preflight",
    }.get(str(row.get("status")), str(row.get("status") or "unknown"))

def validate_rows(rows, *, expected_count, expected_split):
    assert len(rows) == expected_count, (len(rows), expected_count)
    ids = [row_id(row) for row in rows]
    assert len(ids) == len(set(ids)), "Duplicate row IDs"
    for row in rows:
        assert row["split"] == expected_split
        assert canonical_hash(row["messages"]) == row["conversation_sha256"]
        freeze = row.get("silver_freeze")
        if isinstance(freeze, dict):
            assert freeze.get("trainable_in_silver") is True
            assert isinstance(freeze.get("release_authority"), bool)
        else:
            assert row.get("status") in {
                "accepted_expedited_silver",
                "accepted_unanimous_review",
                "accepted_grounded_v3",
            }, (row_id(row), row.get("status"))
        assert row["messages"][0] == {"role": "system", "content": SYSTEM_MESSAGE}
        assert row["messages"][-1]["role"] == "assistant"
        expected_role = "user"
        for message in row["messages"][1:]:
            assert set(message) == {"role", "content"}
            assert message["role"] == expected_role, (row_id(row), expected_role)
            assert isinstance(message["content"], str) and message["content"].strip()
            expected_role = "assistant" if expected_role == "user" else "user"
        assert isinstance(row.get("split_group_sha256"), str)

rows_by_key = {key: read_jsonl(path) for key, path in DATA_PATHS.items()}
for key, rows in rows_by_key.items():
    validate_rows(
        rows,
        expected_count=EXPECTED_ROWS[key],
        expected_split="validation" if key.endswith("validation") else "train",
    )

grounded_train = rows_by_key["grounded120_train"]
grounded_validation = rows_by_key["grounded120_validation"]
complete_train = rows_by_key["complete600_train"]
complete_validation = rows_by_key["complete600_validation"]

def assert_split_isolation(train_rows, validation_rows):
    assert {row_id(row) for row in train_rows}.isdisjoint({row_id(row) for row in validation_rows})
    assert {row["split_group_sha256"] for row in train_rows}.isdisjoint(
        {row["split_group_sha256"] for row in validation_rows}
    )

assert_split_isolation(grounded_train, grounded_validation)
assert_split_isolation(complete_train, complete_validation)
grounded_all = grounded_train + grounded_validation
complete_all = complete_train + complete_validation
assert all(row["corpus_role"] == "grounded" for row in grounded_all)
assert {row["corpus_role"] for row in complete_all} == {"grounded", "revised"}
assert Counter(row["expression_class"] for row in grounded_all) == {
    "shared_implicit": 102, "explicit_sparse": 18,
}
assert Counter(row["provenance"]["source_family"] for row in grounded_all) == {
    "MathDial": 64, "ConvoLearn": 35, "Arabic YouTube academic lesson": 21,
}
assert Counter(row["expression_class"] for row in complete_all) == {
    "shared_implicit": 510, "explicit_sparse": 90,
}
assert sum(row["corpus_role"] == "grounded" for row in complete_train) == 108
assert sum(row["corpus_role"] == "grounded" for row in complete_validation) == 12
assert Counter(row.get("status") for row in complete_all) == Counter(release_manifest["status_counts"])
assert Counter(canonical_tier(row) for row in complete_all) == Counter(release_manifest["tier_counts"])

grounded_index = {
    row_id(row): (row["conversation_sha256"], row["split"], row["split_group_sha256"])
    for row in grounded_all
}
complete_index = {
    row_id(row): (row["conversation_sha256"], row["split"], row["split_group_sha256"])
    for row in complete_all
}
assert grounded_index.keys() <= complete_index.keys()
assert all(complete_index[key] == value for key, value in grounded_index.items())
print("EXACT SILVER RELEASE VALID: 108/12 grounded and 540/60 complete; lineage overlap zero")
print("Tier counts:", json.dumps(release_manifest["tier_counts"], indent=2))

## 3. Validate the locked training controls

In [ ]:
BASE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
BASE_MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
MAX_SEQ_LENGTH = 4096
MAX_STEPS = 75
PER_DEVICE_BATCH = 1
GRADIENT_ACCUMULATION = 8
SAMPLED_CONVERSATIONS = MAX_STEPS * PER_DEVICE_BATCH * GRADIENT_ACCUMULATION
SEED = 3407

LOCKED_CONFIG = {
    "base_model": BASE_MODEL_ID,
    "base_model_revision": BASE_MODEL_REVISION,
    "max_sequence_length": MAX_SEQ_LENGTH,
    "max_steps": MAX_STEPS,
    "effective_batch": PER_DEVICE_BATCH * GRADIENT_ACCUMULATION,
    "sampled_conversations": SAMPLED_CONVERSATIONS,
    "lora_rank": 16,
    "lora_alpha": 32,
    "learning_rate": 2e-4,
    "response_only": True,
    "packing": False,
    "seed": SEED,
}
assert SAMPLED_CONVERSATIONS == 600
print(json.dumps(LOCKED_CONFIG, indent=2))

In [ ]:
def deterministic_cycle(rows, target, namespace):
    assert rows and target >= len(rows)
    ordered = sorted(
        rows,
        key=lambda row: hashlib.sha256(
            f"{SEED}|{namespace}|{row_id(row)}".encode("utf-8")
        ).hexdigest(),
    )
    tagged = [(ordered[index % len(ordered)], index // len(ordered)) for index in range(target)]
    exposures = [
        row for row, occurrence in sorted(
            tagged,
            key=lambda item: hashlib.sha256(
                f"{SEED}|{namespace}|mix|{row_id(item[0])}|{item[1]}".encode("utf-8")
            ).hexdigest(),
        )
    ]
    counts = Counter(row_id(row) for row in exposures)
    assert set(counts) == {row_id(row) for row in rows}
    assert len(exposures) == target
    assert sum(counts.values()) == target
    assert max(counts.values()) - min(counts.values()) <= 1
    assert set(counts.values()) <= {target // len(rows), math.ceil(target / len(rows))}
    exposure_order_sha256 = hashlib.sha256(
        "\n".join(row_id(row) for row in exposures).encode("utf-8")
    ).hexdigest()
    return exposures, dict(sorted(counts.items())), exposure_order_sha256

def load_fresh_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        revision=BASE_MODEL_REVISION,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        full_finetuning=False,
        use_exact_model_name=True,
    )
    assert getattr(model.config, "_commit_hash", None) == BASE_MODEL_REVISION
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )
    return model, tokenizer

def render(rows, tokenizer):
    texts = [
        tokenizer.apply_chat_template(
            row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False
        )
        for row in rows
    ]
    lengths = [
        len(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])
        for text in texts
    ]
    assert max(lengths) <= MAX_SEQ_LENGTH, (max(lengths), MAX_SEQ_LENGTH)
    return Dataset.from_dict({"text": texts}), lengths

## 4. Train the selected Complete-600 adapter

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

RUNS_CONFIG = {
    "grounded120": {
        "run_name": "qwen3_4b_uae_adab_grounded_120_exact_silver_v1",
        "train": grounded_train,
        "validation": grounded_validation,
        "train_path": DATA_PATHS["grounded120_train"],
        "validation_path": DATA_PATHS["grounded120_validation"],
    },
    "complete600": {
        "run_name": "qwen3_4b_uae_adab_complete_600_exact_silver_v1",
        "train": complete_train,
        "validation": complete_validation,
        "train_path": DATA_PATHS["complete600_train"],
        "validation_path": DATA_PATHS["complete600_validation"],
    },
}
assert len({config["run_name"] for config in RUNS_CONFIG.values()}) == 2

def train_selected(config_key):
    global _EXACT_TRAINING_SESSION_RUN
    assert config_key in RUNS_CONFIG
    config = RUNS_CONFIG[config_key]
    adapter_path = RUNS / config["run_name"]
    metrics_path = METRICS / f"{config['run_name']}.json"
    checkpoint_path = RUNS / f"{config['run_name']}_checkpoints"
    assert _EXACT_TRAINING_SESSION_RUN is None, (
        "This A100 session already trained an adapter. Release it and reconnect "
        "before training the other corpus."
    )
    assert not adapter_path.exists(), f"Refusing to overwrite {adapter_path}"
    assert not metrics_path.exists(), f"Refusing to overwrite {metrics_path}"
    assert not checkpoint_path.exists(), f"Refusing to reuse {checkpoint_path}"
    exposures, exposure_counts, exposure_order_sha256 = deterministic_cycle(
        config["train"], SAMPLED_CONVERSATIONS, config_key
    )
    assert len(exposures) == SAMPLED_CONVERSATIONS == 600
    validation_ids = {row_id(row) for row in config["validation"]}
    assert not (set(exposure_counts) & validation_ids)

    _EXACT_TRAINING_SESSION_RUN = config_key
    model, tokenizer = load_fresh_model()
    train_dataset, train_lengths = render(exposures, tokenizer)
    validation_dataset, validation_lengths = render(config["validation"], tokenizer)
    assert len(train_dataset) == SAMPLED_CONVERSATIONS
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        args=SFTConfig(
            dataset_text_field="text",
            max_length=MAX_SEQ_LENGTH,
            per_device_train_batch_size=PER_DEVICE_BATCH,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION,
            warmup_ratio=0.1,
            max_steps=MAX_STEPS,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=10,
            save_strategy="no",
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=SEED,
            data_seed=SEED,
            output_dir=str(checkpoint_path),
            report_to="none",
            packing=False,
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    started = time.monotonic()
    stats = trainer.train()
    losses = [entry["loss"] for entry in trainer.state.log_history if isinstance(entry.get("loss"), (int, float))]
    validation_losses = [
        {"step": entry.get("step"), "eval_loss": entry["eval_loss"]}
        for entry in trainer.state.log_history
        if isinstance(entry.get("eval_loss"), (int, float))
    ]
    assert trainer.state.global_step == MAX_STEPS
    assert losses and validation_losses
    assert all(math.isfinite(value) for value in losses)
    assert all(math.isfinite(item["eval_loss"]) for item in validation_losses)

    model.save_pretrained(str(adapter_path))
    tokenizer.save_pretrained(str(adapter_path))
    adapter_config_path = adapter_path / "adapter_config.json"
    adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
    assert adapter_config["base_model_name_or_path"] == BASE_MODEL_ID
    adapter_config["revision"] = BASE_MODEL_REVISION
    adapter_config_path.write_text(
        json.dumps(adapter_config, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    metrics = {
        "schema_version": "uae_adab_exact_silver_training_metrics.v1",
        "run_name": config["run_name"],
        "train_rows": len(config["train"]),
        "validation_rows": len(config["validation"]),
        "training_exposures": len(exposures),
        "per_assignment_exposure_counts": exposure_counts,
        "exposure_count_min": min(exposure_counts.values()),
        "exposure_count_max": max(exposure_counts.values()),
        "exposure_order_sha256": exposure_order_sha256,
        "validation_overlap": 0,
        "train_dataset_sha256": sha256_file(config["train_path"]),
        "validation_dataset_sha256": sha256_file(config["validation_path"]),
        "release_manifest_sha256": release_manifest_sha256,
        "input_file_sha256": observed_file_sha256,
        "adapter_config_sha256": sha256_file(adapter_config_path),
        "locked_config": LOCKED_CONFIG,
        "fresh_session_guard": {
            "session_nonce_sha256": _EXACT_SESSION_NONCE_SHA256,
            "only_training_run": _EXACT_TRAINING_SESSION_RUN,
        },
        "global_steps": trainer.state.global_step,
        "losses": losses,
        "validation_losses": validation_losses,
        "train_runtime": stats.metrics.get("train_runtime"),
        "wall_seconds": round(time.monotonic() - started, 3),
        "train_token_max": max(train_lengths),
        "validation_token_max": max(validation_lengths),
        "gpu": GPU_NAME,
    }
    metrics_path.write_text(
        json.dumps(metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    print("SAVED", adapter_path, metrics_path)
    del trainer, model, tokenizer, train_dataset, validation_dataset
    gc.collect()
    torch.cuda.empty_cache()
    return adapter_path, metrics_path

RUN_EXPERIMENT = "complete600"
print("Selected final experiment:", RUN_EXPERIMENT)
selected_adapter, selected_metrics = train_selected(RUN_EXPERIMENT)

## 5. Package the adapter and metrics

In [ ]:
RUN_NAME = "qwen3_4b_uae_adab_complete_600_exact_silver_v1"
assert (RUNS / RUN_NAME / "adapter_config.json").is_file()
assert (METRICS / f"{RUN_NAME}.json").is_file()

BUNDLE_ROOT = Path("/content/exact_silver_v1_reproduction_bundle")
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
(BUNDLE_ROOT / "runs_v2_exact").mkdir(parents=True)
(BUNDLE_ROOT / "metrics_v2_exact").mkdir(parents=True)
shutil.copytree(RUNS / RUN_NAME, BUNDLE_ROOT / "runs_v2_exact" / RUN_NAME)
shutil.copy2(
    METRICS / f"{RUN_NAME}.json",
    BUNDLE_ROOT / "metrics_v2_exact" / f"{RUN_NAME}.json",
)
BUNDLE_ZIP = Path(shutil.make_archive(
    "/content/qwen3_4b_uae_adab_exact_silver_v1_reproduction",
    "zip",
    BUNDLE_ROOT,
))
print("REPRODUCTION BUNDLE READY:", BUNDLE_ZIP)
print("Bundle SHA-256:", hashlib.sha256(BUNDLE_ZIP.read_bytes()).hexdigest())

DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    files.download(str(BUNDLE_ZIP))
else:
    print("Set DOWNLOAD_RESULTS=True and rerun this cell to download the ZIP.")